# Space2Soil: Exploratory Agricultural Data Analysis
### NASA Earth Observation Climatology for Ejura, Ashanti Region, Ghana
**Project:** Space2Soil (NASA Space Apps Challenge: "Field Shift: Adapting Farms with NASA Data")  
**Coordinates:** Latitude `7.3856° N`, Longitude `-1.3562° W`  
**Baseline Window:** 2016 – 2025 (10-Year Decadal Record)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Analysis-Ready Datasets

In [ ]:
base_dir = os.path.abspath('..')

# Load clean monthly aggregations
df_climate = pd.read_csv(os.path.join(base_dir, 'data', 'processed', 'climate', 'nasa_power_ejura_monthly_clean.csv'))
df_rainfall = pd.read_csv(os.path.join(base_dir, 'data', 'processed', 'rainfall', 'gpm_ejura_monthly_clean.csv'))
df_soil = pd.read_csv(os.path.join(base_dir, 'data', 'processed', 'soil_moisture', 'smap_ejura_monthly_clean.csv'))
df_veg = pd.read_csv(os.path.join(base_dir, 'data', 'processed', 'vegetation', 'landsat_ejura_monthly_vegetation.csv'))

print('Climate records (months):', len(df_climate))
print('GPM Rainfall records (months):', len(df_rainfall))
print('Soil Moisture records (months):', len(df_soil))
print('Landsat Vegetation records (months):', len(df_veg))

## 2. Decadal Agroclimatological Profile of Ejura, Ghana

In [ ]:
# Merge datasets on year_month
merged = df_climate[['year_month', 'temperature_mean_c', 'temperature_max_extreme_c', 'precipitation_total_mm']].merge(
    df_soil[['year_month', 'surface_wetness_mean', 'rootzone_wetness_mean', 'drought_days_count']], on='year_month'
)
merged = merged.merge(df_veg[['year_month', 'ndvi_mean', 'season']], on='year_month', how='left')
merged['month'] = merged['year_month'].str[5:7].astype(int)

# Compute 10-year monthly climatology
climatology = merged.groupby('month').agg({
    'precipitation_total_mm': 'mean',
    'temperature_mean_c': 'mean',
    'temperature_max_extreme_c': 'max',
    'surface_wetness_mean': 'mean',
    'rootzone_wetness_mean': 'mean',
    'ndvi_mean': 'mean'
}).round(2)

climatology

## 3. Visualizing Key Agricultural Dynamics

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

months = np.arange(1, 13)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Rainfall bars
color = '#1f77b4'
ax1.set_xlabel('Month of Year', fontweight='bold')
ax1.set_ylabel('Mean Monthly Rainfall (mm)', color=color, fontweight='bold')
bars = ax1.bar(months - 0.15, climatology['precipitation_total_mm'], width=0.3, color=color, alpha=0.7, label='Precipitation (mm)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(months)
ax1.set_xticklabels(month_labels)

# Soil moisture and NDVI on secondary axis
ax2 = ax1.twinx()
color_sm = '#8c564b'
color_ndvi = '#2ca02c'
ax2.set_ylabel('Fraction / Index (0.0 - 1.0)', fontweight='bold')
line1 = ax2.plot(months, climatology['rootzone_wetness_mean'], color=color_sm, marker='o', linewidth=2.5, label='Root-Zone Soil Wetness (0-100cm)')
line2 = ax2.plot(months, climatology['ndvi_mean'], color=color_ndvi, marker='s', linewidth=2.5, linestyle='--', label='Landsat NDVI (30m Canopy)')
ax2.set_ylim(0.1, 1.0)

plt.title('Bimodal Cropping Seasons & Climatology in Ejura, Ghana (2016-2025 Average)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()